# Bài 6
Đây là notebook chứa mã nguồn đầy đủ của bài 6.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def _entropy_weights(matrix):
    m, n = matrix.shape
    col_sums = matrix.sum(axis=0) + 1e-9
    P = matrix / col_sums
    P = np.clip(P, 1e-9, 1)
    E = -1 / np.log(m) * np.sum(P * np.log(P), axis=0)
    d = 1 - E
    return d / (d.sum() + 1e-9)

In [ ]:
def solve_bai06(data_dir=None, w_manual=None, weight_mode=0):
    data = get_data(data_dir)
    regions  = data.regions_names_vi.tolist()
    
    # Extract features matching the Bai 6 requirements
    X = data.X_regions.astype(float)
    is_benefit = [True, True, True, True, True, False]
    
    # Calculate baseline weights
    w_entropy = _entropy_weights(X)
    w_expert = np.array(w_manual if w_manual else [0.20, 0.20, 0.20, 0.15, 0.15, 0.10], float)
    w_baseline = w_entropy if weight_mode < 0.5 else w_expert

    def run_topsis(w):
        norm = np.sqrt((X**2).sum(axis=0))
        R = X / (norm + 1e-9)
        V = R * w
        ideal = np.zeros(6)
        anti_ideal = np.zeros(6)
        for j in range(6):
            if is_benefit[j]:
                ideal[j]      = V[:, j].max()
                anti_ideal[j] = V[:, j].min()
            else:
                ideal[j]      = V[:, j].min()
                anti_ideal[j] = V[:, j].max()
        D_plus  = np.sqrt(((V - ideal)**2).sum(axis=1))
        D_minus = np.sqrt(((V - anti_ideal)**2).sum(axis=1))
        C = D_minus / (D_plus + D_minus + 1e-9)
        return C

    C_base = run_topsis(w_baseline)
    ranking_base = np.argsort(-C_base)
    
    result = []
    for rank, i in enumerate(ranking_base):
        result.append({
            'region_name_vi': regions[i],
            'TOPSIS':         round(float(C_base[i]), 4),
            'rank':           rank + 1,
        })

    # Sensitivity analysis: w_AI (index 2) varies from 0.10 to 0.40
    sensitivity_results = {}
    for w_ai_val in np.arange(0.10, 0.45, 0.05):
        w_new = w_baseline.copy()
        w_new[2] = w_ai_val
        # normalize remaining weights so sum is 1
        rem_sum = w_new.sum() - w_new[2]
        if rem_sum > 0:
            for j in range(6):
                if j != 2:
                    w_new[j] = w_new[j] / rem_sum * (1 - w_ai_val)
        
        C_sens = run_topsis(w_new)
        sensitivity_results[f"{w_ai_val:.2f}"] = {regions[i]: round(float(C_sens[i]), 4) for i in range(len(regions))}

    # AHP Simple: using linear additive weighted sum of Min-Max normalized matrix
    def run_ahp(w):
        X_norm = np.zeros_like(X)
        for j in range(6):
            min_v, max_v = X[:,j].min(), X[:,j].max()
            if is_benefit[j]:
                X_norm[:,j] = (X[:,j] - min_v) / (max_v - min_v + 1e-9)
            else:
                X_norm[:,j] = (max_v - X[:,j]) / (max_v - min_v + 1e-9)
        score = X_norm @ w
        return score
        
    ahp_score = run_ahp(w_baseline)
    ranking_ahp = np.argsort(-ahp_score)
    ahp_ranks = {regions[i]: int(np.where(ranking_ahp == i)[0][0] + 1) for i in range(len(regions))}
    topsis_ranks = {regions[i]: int(np.where(ranking_base == i)[0][0] + 1) for i in range(len(regions))}

    return {
        'ranking':    result,
        'weights':    w_baseline.tolist(),
        'closeness':  {regions[i]: round(float(C_base[i]), 4) for i in range(len(regions))},
        'sensitivity': sensitivity_results,
        'ranks_comparison': {
            'TOPSIS': topsis_ranks,
            'AHP': ahp_ranks
        }
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai06()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())